# 03 · Model Interpretability
Grad-CAM is a computer-vision technique that needs a spatial convolutional feature map — it has no direct analogue for a recurrent text classifier. This notebook uses the standard text-model substitute instead: **occlusion / leave-one-out importance** (`src/evaluation/explainability.py`), the same method that powers the Streamlit app's Explainability page.

In [1]:
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from src.inference.predict import SentimentPredictor

predictor = SentimentPredictor()

I0000 00:00:1789174946.586335     614 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1789174946.629479     614 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI AVX512_BF16 AVX512_FP16 AVX_VNNI AMX_TILE AMX_INT8 AMX_BF16 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.


I0000 00:00:1789174948.429877     614 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.


2026-09-12 01:02:30,373 | INFO     | src.inference.predict | Loading LSTM model from /home/claude/work/airline-sentiment-lstm/models/lstm_sentiment.keras


2026-09-12 01:02:30,850 | INFO     | src.inference.predict | Predictor ready (vocab size=10975).


## How occlusion importance works
For a tweet with $n$ tokens, the model scores the full tweet once, then scores $n$ variants — each with exactly one token removed. The drop in the predicted class's probability when a token is removed is that token's importance: large drop = the model relied heavily on that word.

In [2]:
examples = [
    'Thank you so much for getting me on an earlier flight, you saved my trip!',
    'My flight has been delayed 6 hours and nobody will explain why.',
    'The flight was fine, nothing special either way.',
]

for text in examples:
    result = predictor.predict(text, explain=True)
    print(f'\n"{text}"')
    print(f'  -> {result.predicted_class} ({result.confidence:.1%})')
    ranked = sorted(zip(result.tokens, result.token_importances), key=lambda kv: -abs(kv[1]))
    for tok, imp in ranked[:5]:
        print(f'     {tok:>12s}  {imp:+.3f}')

E0000 00:00:1789174951.204506     614 util.cc:131] oneDNN supports DT_BOOL only on platforms with AVX-512. Falling back to the default Eigen-based implementation if present.



"Thank you so much for getting me on an earlier flight, you saved my trip!"
  -> positive (97.2%)
            thank  +0.146
            saved  +0.026
             trip  +0.021
             much  +0.018
          getting  -0.010



"My flight has been delayed 6 hours and nobody will explain why."
  -> negative (99.6%)
            hours  +0.014
          delayed  +0.014
              why  +0.006
             been  +0.004
                6  +0.003

"The flight was fine, nothing special either way."
  -> neutral (48.1%)
              way  +0.169
           either  +0.150
          special  -0.116
              was  -0.111
             fine  -0.079


## Misclassified example walkthrough
Pulling a genuine test-set tweet the LSTM got wrong, to see *why* via the same importance scores — this is the kind of error analysis that separates a portfolio project from a plain accuracy number.

In [3]:
import pandas as pd
from src.config import SPLITS_DIR, LABEL_COLUMN

test_df = pd.read_parquet(SPLITS_DIR / 'test.parquet')
sample = test_df.sample(200, random_state=7)

mistakes = []
for _, row in sample.iterrows():
    pred = predictor.predict(row['clean_text'])
    if pred.predicted_class != row[LABEL_COLUMN]:
        mistakes.append((row['clean_text'], row[LABEL_COLUMN], pred.predicted_class, pred.confidence))

print(f'{len(mistakes)} / {len(sample)} sampled tweets misclassified')
mistakes[:5]

41 / 200 sampled tweets misclassified


[('can i get home before 12 or no', 'negative', 'neutral', 0.8459511995315552),
 ('kid just wants fucking money scumbag',
  'negative',
  'neutral',
  0.5767689347267151),
 ('is there a way to reserve my dog s flight without speaking with a representative just was booted from your helpline',
  'negative',
  'neutral',
  0.7082718014717102),
 ("we're home you guys recovered now we can laugh about it and the extra day in barbados will you open cuba soon",
  'negative',
  'neutral',
  0.7700564861297607),
 ("still haven't been able to get through thanks for responding",
  'neutral',
  'positive',
  0.9368112683296204)]

In [4]:
if mistakes:
    text, true_label, pred_label, conf = mistakes[0]
    print(f'True: {true_label} | Predicted: {pred_label} ({conf:.1%})')
    print(f'Text: {text}')
    result = predictor.predict(text, explain=True)
    ranked = sorted(zip(result.tokens, result.token_importances), key=lambda kv: -abs(kv[1]))
    for tok, imp in ranked:
        print(f'  {tok:>12s}  {imp:+.3f}')

True: negative | Predicted: neutral (84.6%)
Text: can i get home before 12 or no


           can  +0.409
            no  -0.097
            or  +0.086
            12  -0.040
           get  -0.038
        before  +0.026
          home  -0.016
             i  -0.001


## Takeaway
The importance scores consistently surface the words a human would point to as the sentiment-bearing terms (e.g. *delayed*, *rude*, *thank you*, *saved*), which is a useful sanity check that the model has learned genuine sentiment signal rather than some spurious correlation (like a specific airline name) — reinforced by the fact that airline handles are stripped from the text during cleaning (`src/data/text_cleaning.py`).